# Stage 9 — LLM ICD Confirmation

Reviews Stage 8 map packages and **fills gaps**:

1. **Confirm / drop / replace** principal codes (catch bad maps)
2. **Keep** Stage 8 codes that belong on a billing-style list
3. **Name missing conditions** (English) — pipeline maps them via SNOMED → ExtendedMap

The LLM **does not invent ICD strings**. Current-stay ground truth is **not** provided.

**Input:** Stage 8 `icd_coding.json` + clinical context / IE / prior ICDs  
**Output:** `data/stage_09_icd_confirmation/` + per-admission `icd_coding_confirmed.json` / `.txt`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))
REPO = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR

from pipeline import (
    EXPORT_DIR,
    ICD_CONFIRM_CHECKPOINT_JSON,
    ICD_CONFIRM_RESULTS_JSON,
    ICD_CONFIRM_TEMPERATURE,
    LLMNotAvailableError,
    STAGE_09_DIR,
    check_llm,
    get_llm_config,
    print_pipeline_banner,
    run_stage09_cohort,
    warn_if_slow_model,
)
from snomed_ct import build_icd10cm_map_index, build_snomed_index, find_snomed_root

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
warn_if_slow_model(model_info, LLM_CONFIG.provider)
print(f"LLM ready — {LLM_CONFIG.method_prefix()}: {model_info}")
print(f"Confirm temperature: {ICD_CONFIRM_TEMPERATURE}")
STAGE_09_DIR.mkdir(parents=True, exist_ok=True)
print(f"Export dir : {EXPORT_DIR}")
print(f"Stage 9 out: {STAGE_09_DIR}")
print(f"Checkpoint : {ICD_CONFIRM_CHECKPOINT_JSON}")
print("Delete the checkpoint to re-run all admissions.")

In [ ]:
snomed_root = find_snomed_root(REPO / "data")
snomed_index = build_snomed_index(
    snomed_root=snomed_root,
    cache_path=REPO / "data" / "snomed_index" / "snomed_index.pkl",
    force_rebuild=False,
)
map_index = build_icd10cm_map_index(
    snomed_root=snomed_root,
    cache_path=REPO / "data" / "snomed_index" / "icd10cm_extended_map.pkl",
    force_rebuild=False,
    load_titles=True,
)
print(f"SNOMED concepts: {len(snomed_index.active_concepts):,}")
print(f"ICD mapped concepts: {len(map_index.concept_to_maps):,}")

In [ ]:
payload = run_stage09_cohort(
    export_dir=EXPORT_DIR,
    snomed_index=snomed_index,
    map_index=map_index,
    config=LLM_CONFIG,
    temperature=ICD_CONFIRM_TEMPERATURE,
)
print(f"Admissions: {payload.get('n_admissions')}")
print(f"Aggregate : {ICD_CONFIRM_RESULTS_JSON}")
print(f"Per-admission: {EXPORT_DIR}/patient_*/admissions/hadm_*/icd_coding_confirmed.*")

In [ ]:
for row in payload.get("results") or []:
    if row.get("final_codes"):
        print(f"Patient {row.get('patient_id')} HADM {row.get('hadm_id')}")
        print(f"Primary: {row.get('primary_condition')} → {row.get('primary_icd_code')}")
        print(f"Added={row.get('n_added')} dropped={row.get('n_dropped')} fallback={row.get('fallback')}")
        print(row.get("review_summary", "")[:400])
        print("\nFinal codes:")
        for c in row["final_codes"]:
            print(f"  [{c.get('role')}] {c.get('code')} — {c.get('title')}  ({c.get('source')})")
        break
else:
    print("No confirmed packages yet.")